# Triagegeist — Triage Acuity Prediction

**Competition**: [Triagegeist](https://www.kaggle.com/competitions/triagegeist) — Predict emergency severity to support triage decisions (Laitinen-Fredriksson Foundation).

**Target**: `triage_acuity` (1 = most urgent, 5 = least urgent), from structured intake data and optional chief complaint text.

**Data**: Train/test CSVs + `chief_complaints.csv` (NLP) and `patient_history.csv` (comorbidities). All datasets used must be cited per competition rules.

In [13]:
# Setup and load data — works locally (1/data) and on Kaggle (/kaggle/input/...)
import os
import numpy as np
import pandas as pd
from pathlib import Path

if os.path.exists("/kaggle/input"):
    DATA_DIR = Path("/kaggle/input/triagegeist")
    OUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path("data")
    OUT_DIR = Path(".")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
complaints = pd.read_csv(DATA_DIR / "chief_complaints.csv")
history = pd.read_csv(DATA_DIR / "patient_history.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print("Target distribution (triage_acuity):")
print(train["triage_acuity"].value_counts().sort_index())

Train: (80000, 40)
Test: (20000, 37)
Target distribution (triage_acuity):
triage_acuity
1     3222
2    13439
3    28921
4    23020
5    11398
Name: count, dtype: int64


## EDA

- **Structured features**: vitals (BP, HR, RR, temp, SpO2, GCS, pain, BMI), shock_index, news2_score, demographics, arrival mode, chief_complaint_system, etc.
- **Missing**: some vitals are NaN; we'll impute for modeling.
- **Outcome**: `triage_acuity` 1–5 (ordinal). We can treat as classification or ordinal regression.

In [14]:
# Missingness and key predictors
print("Missing % in train:")
print((train.isna().sum() / len(train) * 100).round(1).sort_values(ascending=False).head(15))

# NEWS2 and acuity (clinical severity)
print("\nMean NEWS2 by triage_acuity:")
print(train.groupby("triage_acuity")["news2_score"].agg(["mean", "count"]))

Missing % in train:
mean_arterial_pressure    5.2
pulse_pressure            5.2
systolic_bp               5.2
shock_index               5.2
diastolic_bp              5.2
respiratory_rate          3.8
temperature_c             0.7
patient_id                0.0
arrival_season            0.0
arrival_month             0.0
arrival_day               0.0
arrival_hour              0.0
arrival_mode              0.0
triage_nurse_id           0.0
site_id                   0.0
dtype: float64

Mean NEWS2 by triage_acuity:
                    mean  count
triage_acuity                  
1              13.637492   3222
2              10.073294  13439
3               2.606376  28921
4               0.672546  23020
5               0.348307  11398


## Preprocessing and baseline model

- Drop identifiers and target (and disposition/ed_los_hours on train).
- Encode categoricals; impute numeric missing with median.
- Optional: merge chief_complaint_system (already in main table) or use chief_complaint_raw for NLP later.
- Baseline: ordinal classification with a tree-based model (e.g. LightGBM or XGBoost).

In [15]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import cohen_kappa_score
import warnings
warnings.filterwarnings("ignore")

# Features: exclude ids, target, and outcome-only columns (disposition, ed_los_hours)
id_col = "patient_id"
target_col = "triage_acuity"
drop_cols = [id_col, target_col, "disposition", "ed_los_hours"]

cat_cols = [c for c in train.columns if train[c].dtype == "object" and c not in drop_cols]
num_cols = [c for c in train.columns if c not in drop_cols and train[c].dtype in ["int64", "float64"]]

def prepare(X, encoders=None, fit=False):
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors="ignore")
    out = X.copy()
    if encoders is None:
        encoders = {}
    for col in cat_cols:
        if col not in X.columns:
            continue
        ser = X[col].astype(str).fillna("__NA__")
        if fit:
            encoders[col] = LabelEncoder()
            out[col] = encoders[col].fit_transform(ser).astype(np.int32)
        else:
            classes = set(encoders[col].classes_)
            out[col] = ser.apply(lambda x: encoders[col].transform([x])[0] if x in classes else -1).astype(np.int32)
    for col in num_cols:
        if col in out.columns and out[col].isna().any():
            med = out[col].median()
            out[col] = out[col].fillna(med)
    # LightGBM/sklearn require int, float, or bool — convert any non-numeric column
    # (cat_cols may be string dtype or object; catch all)
    for col in out.columns:
        if not pd.api.types.is_numeric_dtype(out[col]):
            out[col] = pd.to_numeric(out[col], errors="coerce").fillna(-1).astype(np.int32)
    return out, encoders

X_train = train.drop(columns=[target_col])
y_train = train[target_col]
X_train_enc, encoders = prepare(X_train, fit=True)
X_test_enc, _ = prepare(test, encoders=encoders)

# Align columns
for c in X_train_enc.columns:
    if c not in X_test_enc.columns:
        X_test_enc[c] = 0
X_test_enc = X_test_enc[X_train_enc.columns]

# Final dtype guarantee for the model (LightGBM requires int/float/bool)
X_train_enc = X_train_enc.astype(np.float32)
X_test_enc = X_test_enc.astype(np.float32)

print("Feature matrix shape:", X_train_enc.shape)
print("Dtypes:", X_train_enc.dtypes.unique())
print("Target balance:", y_train.value_counts().sort_index().tolist())

Feature matrix shape: (80000, 36)
Dtypes: [dtype('float32')]
Target balance: [3222, 13439, 28921, 23020, 11398]


In [16]:
# Baseline: LightGBM multiclass (or use XGBoost)
try:
    import lightgbm as lgb
    use_lgb = True
except ImportError:
    use_lgb = False

if use_lgb:
    model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    preds_cv = cross_val_predict(model, X_train_enc, y_train, cv=cv, method="predict")
    kappa = cohen_kappa_score(y_train, preds_cv, weights="quadratic")
    print(f"CV Quadratic Cohen's Kappa: {kappa:.4f}")
    model.fit(X_train_enc, y_train)
    test_pred = model.predict(X_test_enc)
else:
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train_enc, y_train)
    test_pred = model.predict(X_test_enc)
    print("Using RandomForest (install lightgbm for better performance)")

CV Quadratic Cohen's Kappa: 0.9253


In [17]:
# Build submission (patient_id, triage_acuity)
sub = pd.DataFrame({
    "patient_id": test["patient_id"],
    "triage_acuity": test_pred.astype(int)
})
# Clamp to valid range 1-5
sub["triage_acuity"] = sub["triage_acuity"].clip(1, 5)
sub.to_csv(OUT_DIR / "submission.csv", index=False)
print(sub.head(10))
print("\nSubmission saved to submission.csv")

     patient_id  triage_acuity
0  TG-FZUFCRZS3              2
1  TG-SSCOXTYI1              3
2  TG-JY74ZR35D              2
3  TG-JDKD5G62X              5
4  TG-J1BSAAXR0              5
5  TG-5YERUAA5X              4
6  TG-IVU18ZJCF              2
7  TG-K5XYMWTCG              2
8  TG-RCAR2TLEI              4
9  TG-YNDUR0KGO              4

Submission saved to submission.csv
